In [ ]:
from proto import *
from engine import *

import pandas as pd
import numpy as np

pd.options.plotting.backend = "plotly"

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

In [ ]:
def foi(cdatamap, params):
    ipop = cdatamap.query([disease_state["I"]]).data.sum()
    total_pop = cdatamap.data.sum()
    return (ipop / total_pop) * params["contact_rate"]

In [ ]:
infection = TransitionFlow(disease_state["S"], disease_state["I"], "foi")
recovery = TransitionFlow(disease_state["I"], disease_state["R"], "recovery_rate")

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {"foi": foi}

model = NaiveModel(humans, flows, dyn_params)
run = model.get_runner()

In [ ]:
istate = jnp.array([100.0, 5.0, 0.0])
params = {"contact_rate": 0.2, "recovery_rate": 0.01}
t = 200

comp_results = run(istate, params, t)

In [ ]:
def compname(c: Compartment):
    return "_".join([stratum for (strat, stratum) in c.strata])


comp_labels = [compname(c) for c in model.cmap.compartments]

In [ ]:
pd.DataFrame(comp_results, columns=comp_labels).plot()

In [ ]:
severity_strat = humans.stratify(
    Stratification("severity", ["mild", "severe"]), disease_state["I"]
)

In [ ]:
humans

In [ ]:
run = model.get_runner()
istate = jnp.array([100.0, 5.0, 0.0, 0.0])
params = {"contact_rate": 0.2, "recovery_rate": 0.01}
t = 200

comp_results = run(istate, params, t)

comp_labels = [compname(c) for c in model.cmap.compartments]
pd.DataFrame(comp_results, columns=comp_labels).plot()

In [ ]:
age_strat = humans.stratify(
    Stratification("age", ["child", "young_adult", "adult", "older"])
)

In [ ]:
inf_age_cats = [
    [disease_state["I"], age_strat[age_group]] for age_group in age_strat.strata
]

inf_age_cats

In [ ]:
mm = jnp.ones((len(age_strat.strata), len(age_strat.strata)))


def foi_mixing(cdatamap, params):
    ipops = query_cat_reduction(inf_age_cats, cdatamap)
    total_pop = cdatamap.data.sum()
    age_foi = mm @ ipops / total_pop * params["contact_rate"]
    return CategoryData(age_strat.categories(), age_foi)

In [ ]:
istate = humans.zeros(np)
s_idx = humans.query(disease_state["S"]).indices
istate.data[s_idx] = np.array((10.0, 20.0, 50.0, 20.0))
i_idx = humans.query(disease_state["I"]).indices
istate.data[i_idx] = 1.0

In [ ]:
foi_mixing(istate, params)

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {"foi": foi_mixing}

model = NaiveModel(humans, flows, dyn_params)

run = model.get_runner()

params = {"contact_rate": 0.2, "recovery_rate": 0.02}
t = 200

comp_results = run(istate.data, params, t)

comp_labels = [compname(c) for c in model.cmap.compartments]
pd.DataFrame(comp_results, columns=comp_labels).plot()